# Session 16: RAGAS Evaluation with Cost Analysis

Compare an open-source Fireworks AI RAG pipeline against an OpenAI `gpt-4.1-mini` equivalent using:
- **RAGAS** metrics: context recall, faithfulness, factual correctness, answer relevancy
- **LangSmith** tracing: token usage and cost per query

## Prerequisites
- `.env` file with `OPENAI_API_KEY`, `FIREWORKS_API_KEY`, `LANGCHAIN_API_KEY`
- PDF data in `data/cat-health-guide.pdf`

## Task 1: Environment Setup

In [15]:
import os
from dotenv import load_dotenv

load_dotenv()

# Override project name so traces are grouped separately for this evaluation
os.environ["LANGSMITH_PROJECT"] = "Session16-RAG-Eval"

## Task 2: Build Two RAG Pipelines

We reuse `_build_rag_graph` from `app/rag.py`, passing different embedding and LLM models.

In [16]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from app.rag import _build_rag_graph

DATA_DIR = "data"

# Pipeline 1: Fireworks AI (open-source models)
fw_graph = _build_rag_graph(DATA_DIR)

# Pipeline 2: OpenAI
oai_embedding = OpenAIEmbeddings(model="text-embedding-3-small")
oai_llm = ChatOpenAI(model="gpt-4.1-mini")
oai_graph = _build_rag_graph(DATA_DIR, embedding_model=oai_embedding, generator_llm=oai_llm)

print("Both RAG pipelines built successfully.")

Both RAG pipelines built successfully.


### Sanity check

In [17]:
test_q = "What vaccinations do cats need?"

fw_resp = fw_graph.invoke({"question": test_q})
print("Fireworks:", fw_resp["response"][:200])

oai_resp = oai_graph.invoke({"question": test_q})
print("\nOpenAI:", oai_resp["response"][:200])

Fireworks: **Core feline vaccines (recommended for all cats)**  
1. **Rabies** (the only vaccine required by law in many areas)  
2. **Feline herpesvirus type 1 (FHV‑1)** – the cause of “cat flu”  
3. **Feline c

OpenAI: Cats need core vaccines which include rabies virus, feline herpesvirus type 1 (FHV-1), feline calicivirus (FCV), and feline panleukopenia virus (FPV). Vaccination against feline leukemia virus (FeLV) 


## Task 3: Synthetic Test Data Generation with RAGAS

Generate synthetic questions from the same PDF used by the RAG pipelines.

We use `generate_with_chunks` instead of `generate_with_langchain_docs` because this academic PDF
lacks clear section headers, causing RAGAS's `HeadlinesExtractor` → `HeadlineSplitter` pipeline to fail.
Pre-chunking bypasses that step while preserving the rest of the SDG pipeline (themes, NER, embeddings, similarity).

In [18]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator

# Load source documents
directory_loader = DirectoryLoader(DATA_DIR, glob="**/*.pdf", loader_cls=PyMuPDFLoader)
docs = directory_loader.load()
print(f"Loaded {len(docs)} document pages.")

# Pre-chunk documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks.")

# SDG uses a separate model to avoid circularity
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_chunks(chunks=chunks, testset_size=10)

Loaded 22 document pages.
Split into 124 chunks.


/var/folders/r4/cqrwpld94ldc4rgfhq1fbx8h0000gn/T/ipykernel_48638/4203258798.py:18: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
/var/folders/r4/cqrwpld94ldc4rgfhq1fbx8h0000gn/T/ipykernel_48638/4203258798.py:19: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


Applying SummaryExtractor:   0%|          | 0/124 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/124 [00:00<?, ?it/s]

Node 4fee697f-f53a-4430-a0f8-1dbb56f21bc7 does not have a summary. Skipping filtering.
Node e9f25944-87f0-45b4-b84d-f2d192bd84a0 does not have a summary. Skipping filtering.
Node 287ae989-73ff-4104-a72a-3d1eb44e24b2 does not have a summary. Skipping filtering.
Node a3ae8346-4642-4980-ac53-be9a480db4d0 does not have a summary. Skipping filtering.
Node 942cb7f3-43c4-49ed-9409-4855e0309f25 does not have a summary. Skipping filtering.
Node 2e92c944-98d4-4194-a65a-eb65070adcf0 does not have a summary. Skipping filtering.
Node 2542a58d-6657-40f2-822a-d59a938cbd14 does not have a summary. Skipping filtering.
Node 8ddecfd2-3fa8-4fa0-9c92-55c1fbaaefdf does not have a summary. Skipping filtering.
Node f48dd928-7523-46ff-9928-d12e6d213c17 does not have a summary. Skipping filtering.
Node ea013c40-4947-486a-9f99-f430ebbe531f does not have a summary. Skipping filtering.
Node d27ec921-7276-491b-ad21-c948d7acbcfd does not have a summary. Skipping filtering.
Node 8e08791f-0705-4ea1-9d9d-2bcf29503c32 d

...

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [19]:
dataset.to_pandas()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,wut r the 2021 AAHA/AAFP Feline Life Stage Gui...,[VETERINARY PRACTICE GUIDELINES\n2021 AAHA/AAF...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,Cat Owner and Caregiver,MISSPELLED,LONG,single_hop_specific_query_synthesizer
1,wat is oral helth for cats?,"[and senior) as well as an end-of-life stage, ...",Oral health is one of the most critical health...,Feline Welfare Consultant,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
2,wut r the 2021 AAHA/AAFP Feline Life Stage Gui...,[medical history; behavior; risk assessment; e...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,Cat Owner and Caregiver,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
3,What does the abbreviation DER stand for in th...,"[course of treatment, or procedure. Variations...",DER stands for daily energy requirements.,Feline Welfare Consultant,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
4,Why cat owners need biosecurity measures if fe...,[<1-hop>\n\npeople” resource.128 This informat...,Cat owners need biosecurity measures if feedin...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
5,Howw doo enviromental stressors like exturnal ...,"[<1-hop>\n\n130. van Bree FPJ, Bokken GCAM, Mi...",Enviromental stressors such as exturnal animal...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
6,How can environmental stressors such as extern...,[<1-hop>\n\nInvestigating Urine Marking\n· Whe...,"Environmental stressors like external animals,...",NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
7,How can owner education help address issues re...,"[<1-hop>\n\nspace, administration at this loca...",Owner education plays a crucial role in addres...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
8,how do i make sure my cat patient has less str...,[<1-hop>\n\nthe cat is reacting to the environ...,To make sure your cat patient has less stress ...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
9,wat r importnt health and behavor things to ch...,[<1-hop>\n\ndetection of changes and identiﬁca...,Important health and behavior things to check ...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer


## Task 4: Run Both Pipelines on Test Data

In [22]:
import copy
import time
from openai import RateLimitError

# --- Fireworks pipeline (with retry for rate limits) ---
fw_dataset = copy.deepcopy(dataset)
for i, test_row in enumerate(fw_dataset):
    for attempt in range(3):
        try:
            response = fw_graph.invoke({"question": test_row.eval_sample.user_input})
            test_row.eval_sample.response = response["response"]
            test_row.eval_sample.retrieved_contexts = [ctx.page_content for ctx in response["context"]]
            print(f"  [{i+1}/{len(fw_dataset.samples)}] done")
            break
        except RateLimitError:
            wait = 30 * (attempt + 1)
            print(f"  [{i+1}/{len(fw_dataset.samples)}] rate limited, waiting {wait}s...")
            time.sleep(wait)
    time.sleep(15)

print("Fireworks pipeline done.")

  [1/12] done
  [2/12] done
  [3/12] done
  [4/12] done
  [5/12] done
  [6/12] done
  [7/12] done
  [8/12] done
  [9/12] done
  [10/12] done
  [11/12] done
  [12/12] done
Fireworks pipeline done.


In [23]:
# --- OpenAI pipeline ---
oai_dataset = copy.deepcopy(dataset)
for i, test_row in enumerate(oai_dataset):
    response = oai_graph.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = response["response"]
    test_row.eval_sample.retrieved_contexts = [ctx.page_content for ctx in response["context"]]
    print(f"  [{i+1}/{len(oai_dataset.samples)}] done")

print("OpenAI pipeline done.")

  [1/12] done
  [2/12] done
  [3/12] done
  [4/12] done
  [5/12] done
  [6/12] done
  [7/12] done
  [8/12] done
  [9/12] done
  [10/12] done
  [11/12] done
  [12/12] done
OpenAI pipeline done.


## Task 5: RAGAS Evaluation

In [26]:
import numpy as np
from ragas import EvaluationDataset, evaluate, RunConfig
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
custom_run_config = RunConfig(timeout=360)
metrics = [LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy()]

fw_df = fw_dataset.to_pandas().replace({np.nan: None})
fw_eval_dataset = EvaluationDataset.from_pandas(fw_df)

fw_result = evaluate(
    dataset=fw_eval_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    run_config=custom_run_config,
)
print("Fireworks RAGAS results:")
fw_result

/var/folders/r4/cqrwpld94ldc4rgfhq1fbx8h0000gn/T/ipykernel_48638/3523268500.py:3: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy
/var/folders/r4/cqrwpld94ldc4rgfhq1fbx8h0000gn/T/ipykernel_48638/3523268500.py:3: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy
/var/folders/r4/cqrwpld94ldc4rgfhq1fbx8h0000gn/T/ipykernel_48638/3523268500.py:3: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please 

Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Fireworks RAGAS results:


{'context_recall': 0.8653, 'faithfulness': 0.5501, 'factual_correctness(mode=f1)': 0.2625, 'answer_relevancy': 0.6907}

In [27]:
oai_df = oai_dataset.to_pandas().replace({np.nan: None})
oai_eval_dataset = EvaluationDataset.from_pandas(oai_df)

oai_result = evaluate(
    dataset=oai_eval_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    run_config=custom_run_config,
)
print("OpenAI RAGAS results:")
oai_result

Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


OpenAI RAGAS results:


{'context_recall': 0.8653, 'faithfulness': 0.8947, 'factual_correctness(mode=f1)': 0.3658, 'answer_relevancy': 0.7840}

In [29]:
import pandas as pd

fw_scores = fw_result.to_pandas().select_dtypes(include="number").mean()
oai_scores = oai_result.to_pandas().select_dtypes(include="number").mean()

comparison = pd.DataFrame({
    "Fireworks (OSS)": fw_scores,
    "OpenAI (gpt-4.1-mini)": oai_scores,
})
comparison["Delta"] = comparison["OpenAI (gpt-4.1-mini)"] - comparison["Fireworks (OSS)"]
comparison

,Fireworks (OSS),OpenAI (gpt-4.1-mini),Delta
context_recall,0.865278,0.865278,0.000000
faithfulness,0.550106,0.894685,0.344579
factual_correctness(mode=f1),0.262500,0.365833,0.103333
answer_relevancy,0.690654,0.784028,0.093374


## Task 6: LangSmith Evaluation with Cost Tracking

Upload the synthetic test data to a LangSmith dataset and run both pipelines through
`langsmith.evaluation.evaluate()` to capture per-query token usage and cost in the dashboard.

In [30]:
from uuid import uuid4
from langsmith import Client

ls_client = Client()

# Create a LangSmith dataset from our synthetic test data
ls_dataset_name = f"Session16-RAG-Eval-{uuid4().hex[:8]}"
ls_dataset = ls_client.create_dataset(ls_dataset_name, description="Synthetic test data for Session 16 RAG comparison")

test_df = dataset.to_pandas()
for _, row in test_df.iterrows():
    ls_client.create_example(
        inputs={"question": row["user_input"]},
        outputs={"reference": row["reference"]},
        dataset_id=ls_dataset.id,
    )

print(f"Created LangSmith dataset '{ls_dataset_name}' with {len(test_df)} examples.")

Created LangSmith dataset 'Session16-RAG-Eval-786e0ef6' with 12 examples.


### Define evaluators

In [33]:
from openevals.llm import create_llm_as_judge

# QA correctness
qa_evaluator = create_llm_as_judge(
    prompt="You are evaluating a QA system. Given the input, assess whether the prediction is correct.\n\nInput: {inputs}\nPrediction: {outputs}\nReference answer: {reference_outputs}\n\nIs the prediction correct? Return 1 if correct, 0 if incorrect.",
    feedback_key="qa",
    model="openai:gpt-4.1-mini",
)

# Labeled helpfulness
labeled_helpfulness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "helpfulness: Is this submission helpful to the user, "
        "taking into account the correct reference answer?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n"
        "Reference answer: {reference_outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="helpfulness",
    model="openai:gpt-4.1-mini",
)

evaluators = [qa_evaluator, labeled_helpfulness_evaluator]

### Define target functions for each pipeline

In [34]:
def fw_target(inputs: dict) -> dict:
    """Run the Fireworks RAG pipeline."""
    result = fw_graph.invoke({"question": inputs["question"]})
    return {"output": result["response"]}

def oai_target(inputs: dict) -> dict:
    """Run the OpenAI RAG pipeline."""
    result = oai_graph.invoke({"question": inputs["question"]})
    return {"output": result["response"]}

### Run LangSmith evaluation for both pipelines

In [35]:
from langsmith.evaluation import evaluate as ls_evaluate

fw_ls_result = ls_evaluate(
    fw_target,
    data=ls_dataset_name,
    evaluators=evaluators,
    metadata={"revision_id": "fireworks_oss"},
)
print("Fireworks LangSmith evaluation complete.")

View the evaluation results for experiment: 'best-thumb-26' at:
https://smith.langchain.com/o/991456bb-c3e2-422e-957d-4e5bfe176b65/datasets/797ed67f-678f-4c25-b401-c9e24ccd97d7/compare?selectedSessions=2e10a5df-d116-465f-b6df-d08479dd313e




0it [00:00, ?it/s]

Fireworks LangSmith evaluation complete.


In [36]:
oai_ls_result = ls_evaluate(
    oai_target,
    data=ls_dataset_name,
    evaluators=evaluators,
    metadata={"revision_id": "openai_gpt41mini"},
)
print("OpenAI LangSmith evaluation complete.")

View the evaluation results for experiment: 'enchanted-rate-58' at:
https://smith.langchain.com/o/991456bb-c3e2-422e-957d-4e5bfe176b65/datasets/797ed67f-678f-4c25-b401-c9e24ccd97d7/compare?selectedSessions=8c509399-3f7d-479c-8983-b0372b3460ed




0it [00:00, ?it/s]

OpenAI LangSmith evaluation complete.


## Task 7: Analysis

### LangSmith Results

![LangSmith Evaluation Dashboard](langsmith.png)

### Combined Comparison

| Metric | Fireworks (OSS) | OpenAI (gpt-4.1-mini) | Delta |
|--------|----------------|----------------------|-------|
| **RAGAS: context_recall** | 0.8653 | 0.8653 | 0.0000 |
| **RAGAS: faithfulness** | 0.5501 | 0.8947 | +0.3446 |
| **RAGAS: factual_correctness** | 0.2625 | 0.3658 | +0.1033 |
| **RAGAS: answer_relevancy** | 0.6907 | 0.7840 | +0.0934 |
| **LangSmith: qa** | 0.67 | 0.75 | +0.08 |
| **LangSmith: helpfulness** | 0.75 | 0.83 | +0.08 |
| **LangSmith: P50 latency** | 5.18s | 2.85s | -2.33s |
| **LangSmith: P99 latency** | 13.23s | 5.81s | -7.42s |
| **LangSmith: cost** | ~$0.008 | ~$0.008 | ~$0.00 |

### Key Takeaways

1. **Context recall is identical** (0.8653) — both embedding models (Fireworks qwen3-embedding-8b vs OpenAI text-embedding-3-small) retrieve equally relevant context for this dataset.
2. **OpenAI outperforms on other RAG metrics** — OpenAI model leads on factual correctness, answer relevancy, QA correctness, and helpfulness, while Fireworks GPT OSS performs much worse, likely due to it hallucinates more
3. **OpenAI is ~2x faster at similar cost** — P50 latency is 2.85s vs 5.18s, and total cost is comparable. OpenAI is the clear winner here: better quality, faster response, and no meaningful cost penalty.